# ComfyUI + Krea 2 Turbo (Public SFW Render-Worker für Google Colab)

**Datenschutz-Hinweis:** Dieses Notebook lädt **ausschließlich öffentliche Dateien** (ComfyUI + `Comfy-Org/Krea-2` von HuggingFace). Es wird kein Google Drive gemountet und es landen keine privaten Fotos oder LoRAs in der Cloud.

### Ablauf
1. Oben rechts **Runtime → Change runtime type → L4 / A100 / H100** wählen.
2. Alle Zellen von oben nach unten ausführen (`Cmd+F9` / `Run all`).
3. Die angezeigte `https://....trycloudflare.com`-URL kopieren und lokal auf dem Mac übergeben:
   ```bash
   python3 tools/baue_location_poster.py --server https://<DEINE-URL>.trycloudflare.com --comfy-org
   ```
4. Wenn das Poster fertig ist: **Runtime → Disconnect and delete runtime**, damit keine Compute Units im Leerlauf verbrennen.

In [ ]:
#@title 1. GPU prüfen & ComfyUI installieren
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!apt-get -qq update && apt-get -qq install -y aria2

import os
if not os.path.exists('/content/ComfyUI'):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
%cd /content/ComfyUI
!pip -q install -r requirements.txt

In [ ]:
#@title 2. Öffentliche Krea 2 Turbo Modelle (Comfy-Org/Krea-2) laden
HF = "https://huggingface.co/Comfy-Org/Krea-2/resolve/main"
MODELS = [
    (f"{HF}/diffusion_models/krea2_turbo_fp8_scaled.safetensors", "/content/ComfyUI/models/diffusion_models", "krea2_turbo_fp8_scaled.safetensors"),
    (f"{HF}/text_encoders/qwen3vl_4b_fp8_scaled.safetensors", "/content/ComfyUI/models/text_encoders", "qwen3vl_4b_fp8_scaled.safetensors"),
    (f"{HF}/vae/qwen_image_vae.safetensors", "/content/ComfyUI/models/vae", "qwen_image_vae.safetensors"),
]

import os, subprocess
for url, dest_dir, fname in MODELS:
    os.makedirs(dest_dir, exist_ok=True)
    target = os.path.join(dest_dir, fname)
    if os.path.exists(target) and os.path.getsize(target) > 10_000_000:
        print(f"Bereits vorhanden: {fname}")
        continue
    print(f"Lade {fname} ...")
    subprocess.run([
        "aria2c", "-x", "16", "-s", "16", "-k", "1M",
        "--console-log-level=warn", "--summary-interval=10",
        "-d", dest_dir, "-o", fname, url
    ], check=True)
print("Alle 3 Krea-2-Modelle bereit!")

In [ ]:
#@title 3. ComfyUI starten & Cloudflare Quick-Tunnel öffnen
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared

import subprocess, threading, time, re, urllib.request

comfy = subprocess.Popen(
    ["python", "main.py", "--listen", "127.0.0.1", "--port", "8188", "--disable-auto-launch"],
    cwd="/content/ComfyUI",
    stdout=open("/content/comfyui.log", "w"),
    stderr=subprocess.STDOUT,
)

# Warten bis ComfyUI auf Port 8188 antwortet
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        print("ComfyUI läuft auf 127.0.0.1:8188")
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError("ComfyUI konnte nicht gestartet werden. Siehe /content/comfyui.log")

tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        print("\n" + "=" * 68)
        print("COLAB COMFYUI TUNNEL BEREIT:")
        print(f"  {url}")
        print("\nAuf deinem Mac ausführen:")
        print(f"  python3 tools/baue_location_poster.py --server {url} --comfy-org")
        print("=" * 68 + "\n")
        break

# Zelle aktiv halten, solange der Tunnel läuft (mit Stopp-Button beenden)
try:
    while tunnel.poll() is None and comfy.poll() is None:
        time.sleep(5)
except KeyboardInterrupt:
    pass
finally:
    tunnel.terminate()
    comfy.terminate()
    print("ComfyUI und Tunnel gestoppt.")